# GOV-01 Version 2 preprocessing

This notebook prepares the Rome Road Damage Dataset clean split for later Version 2 model experiments. It **does not train a model** and it does not evaluate the protected test split.

The binary task is `No_pothole` versus `Pothole`. A `No_pothole` image may still contain a crack or a manhole. The project does not claim that this dataset labels shadows, fresh repairs, road markings, or all other confusing road features.

## Before running

Run `src/audit_road_damage_coco.py` and `src/build_road_damage_rome_split.py` first. The ignored derived split must then be available at `data/processed/road_damage_rome_clean_split/`.

In Google Colab, clone the repository, make the raw Rome dataset available locally, and rebuild the clean split. Raw images and ZIP files are intentionally not stored in GitHub.

In [ ]:
from pathlib import Path
from collections import Counter
import tensorflow as tf

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
DATA_DIR = Path('data/processed/road_damage_rome_clean_split')
CLASS_NAMES = ['No_pothole', 'Pothole']

tf.keras.utils.set_random_seed(SEED)
print('TensorFlow version:', tf.__version__)
print('Clean V2 split:', DATA_DIR.resolve())

In [ ]:
for split_name in ['train', 'validation', 'test']:
    split_path = DATA_DIR / split_name
    if not split_path.is_dir():
        raise FileNotFoundError(f'Missing clean-split folder: {split_path}')
    found_classes = sorted(path.name for path in split_path.iterdir() if path.is_dir())
    if found_classes != CLASS_NAMES:
        raise ValueError(f'{split_name} has {found_classes}; expected {CLASS_NAMES}')

print('Folder structure verified.')
print('The protected test folder exists but will not be loaded in this notebook.')

In [ ]:
def load_model_selection_split(split_name, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        DATA_DIR / split_name,
        labels='inferred',
        label_mode='binary',
        class_names=CLASS_NAMES,
        color_mode='rgb',
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=SEED if shuffle else None,
    )

train_raw = load_model_selection_split('train', shuffle=True)
validation_ds = load_model_selection_split('validation', shuffle=False)

print('Loaded training and validation only.')
print('Class index mapping:', dict(enumerate(CLASS_NAMES)))

In [ ]:
training_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.03),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.10),
], name='v2_training_augmentation')

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_raw.map(
    lambda images, labels: (training_augmentation(images, training=True), labels),
    num_parallel_calls=AUTOTUNE,
).prefetch(AUTOTUNE)
validation_ds = validation_ds.prefetch(AUTOTUNE)

print('Random augmentation is applied only to train_ds.')
print('validation_ds has no random augmentation. The protected test split remains unloaded.')

In [ ]:
images, labels = next(iter(train_ds))
batch_counts = Counter(labels.numpy().astype(int).ravel())

print('Training batch image shape:', images.shape)
print('Training batch label shape:', labels.shape)
print('First training batch counts:', {CLASS_NAMES[key]: value for key, value in batch_counts.items()})
print('Expected image format: 224 x 224 RGB; label 0 = No_pothole, 1 = Pothole.')

## Preprocessing boundary

- The loader resizes every image to 224 x 224 pixels and keeps RGB colour.
- Training images receive small horizontal-flip, rotation, zoom, and contrast variations. These make the model less dependent on one exact camera angle or lighting condition; they do **not** create labels for shadows or repairs.
- Validation images are resized only—no random variation.
- The protected test split is not loaded here. It will be used once only after a V2 candidate is selected.
- Pixel scaling stays inside each future model: `Rescaling(1/255)` for a baseline CNN and `mobilenet_v2.preprocess_input` for MobileNetV2.